# Lab 3: Neural Signal Processing & Decoding

## Introduction

The data we will be analyzing is part of a much larger dataset published in [[1](https://pubmed.ncbi.nlm.nih.gov/24945777/)]. Full details and description of the data collection methods and task can be found in that publication. 

**Subject and neural implants:** A multi-electrode array with 128 electrodes was chronically implanted in the motor/pre-motor cortex of one male rhesus macaque monkey (“Jeeves”). The subject was trained to perform a variety of sensorimotor behavioral tasks prior to surgical implantation. 

**Neural data acquisition:** Neural data was acquired from the 128 implanted electrodes. Recordings were referenced using a local reference electrode placed in the skull near the array but not directly in the cortical tissue. Local-field potentials from each electrode were recorded by filtering low frequency portions of the measured voltage (0 – 150 Hz), sampled at 1 kHz. Spiking activity was detected using a voltage threshold (5 standard deviations from the signal root-mean-square value). When clearly separable spike waveforms were recorded, spikes were sorted using waveform template matching. When waveforms could not be isolated, all detected spikes on a channel were lumped together, constituting “multi-unit” activity capturing activity of multiple neurons. 

**Task:** The monkey performed a delayed a self-paced delayed center-out reaching task to eight targets (Figure 1). Trials were initiated by moving to the central target. A successful trial required a short hold at the center, moving to the peripheral target within a specified time-limit, and a brief hold at the target. Successful trials resulted in a liquid reward; failed trials were repeated. Target directions were presented in a blocked pseudo-randomized order. The subject’s arm moved in a KINARM exoskeleton which restricted movements to the horizontal plane (Figure 1A).  Targets were circular with a 1.2 - 1.7cm radius uniformly distributed around a 13cm diameter circle. 

<p align="center">
  <img src="assets/Fig1.png" alt="Image" width="500">
</p>

## Data Types and Formatting

### Behavioral data
Arm movement kinematics (position, velocity, and acceleration) and task events (category of event, time of occurrence) were recorded simultaneously with neural data. Kinematics were originally collected through the KINARM exoskeleton in joint-based coordinates. For the purposes of this lab, you have been given the kinematics transformed into Cartesian coordinates of the hand (i.e. X and Y position). 

Kinematic data files (*_kinematics.mat) include the following variables:
- hand_kinematics – [time x 6] matrix with time-series of hand kinematics in cartesian coordinates, sampled at 1 kHz
    - Column 1 = hand X position
    - Column 2 = hand Y position
    - Column 3 = hand X velocity
    - Column 4 = hand Y velocity
    - Column 5 = hand X acceleration
    - Column 6 = hand Y acceleration
- hand_kinematics_subsampled – [time x 6] matrix with time-series of hand kinematics in cartesian coordinates, down-sampled to 100 Hz. 
    - Because hand kinematics generally have relatively slow dynamics, we will primarily use the subsampled kinematics for our analyses. 
- EVENTS – [# events x 1] vector of integer “event codes” to denote the identity of each event (e.g. trial start vs. reward delivered). A full list of event codes and their meanings is provided below. 
- EVENT_TIMES – [# events x 1] vector of time-stamps for each task event (in seconds)
- FS_kinematics – double with the value of the kinematics sampling rate
- FS_kinematics_sub – double with the value of the down-sampled kinematics sampling rate

### Neuron spikes
The time-stamps of all detected spikes for each of the 128 electrodes were saved. For the purposes of our analyses, we will be analyzing a subset of electrodes. Spike time-stamps are stored in a variable called spike_times, which contains spike times for all units we will analyze. In this lab we will refer to the spiking activity as “units” (this is a term used in a lot of neuroscience research for measured action potentials where we can’t sort the spikes to identify individual neurons). 

Spike data files (*_spikes.mat) include the following variables:
- spike_times – cell {# units x 1}. Each cell entry contains a vector [1 x #spikes] with the spike time-stamps for that unit
- EVENTS – [# events x 1] vector of integer “event codes” to denote the identity of each event (e.g. trial start vs. reward delivered). A full list of event codes and their meanings is provided below. 
- EVENT_TIMES – [# events x 1] vector of time-stamps for each task event (in seconds)

### Event-codes
The event codes (value of EVENTS variable) signify different types of events that could occur in the center-out task. Here is a list of event-code meanings:
- 2: center target appears (cuing start of a trial)
- 15: hand enters the center target
- 5: go-cue occurs (instructing the subject to start reaching)
- 6: hand leaves the center target
- 7: hand enters the peripheral target
- 9: reward turns on (trial successful)
- 11: trial complete
- 64 – 71: code to note which target is presented (64 = target 1, 65 = target 2, etc.). Targets are numbered counter-clockwise starting at 3 o’clock as shown in Figure 1. 
- 4: error holding at the center (hand left center before the go-cue)
- 8: error holding at the peripheral target (hand left before designated hold time elapsed)
- 12: reach time-out error (did not enter the peripheral target before time-limit elapsed)
- 51: you can ignore this event code—it’s irrelevant to the analyses we will be doing. 
- 4101: task stopped or paused


In [ ]:
# Setup
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from sklearn.metrics import r2_score


rng = np.random.default_rng(0)
lab_dir = Path.cwd() if (Path.cwd()/"data").is_dir() else Path.cwd()/"lab3"
data_dir = lab_dir/"data"
file_base = "jeev050712a_bioe466_winter2020"

## 1. Raw Data Visualization
Again, before start working with the data, let's check their dimension and plot some simple visualizations.

In [ ]:
# TODO: check dimensions of kinematics and spike data

# Load kinematics and spikes data.
kinematic_data = loadmat(data_dir/f"{file_base}_kinematics.mat",squeeze_me=True)
spike_data = loadmat(data_dir/f"{file_base}_spikes.mat",squeeze_me=True)

hand_kinematics = kinematic_data["hand_kinematics_subsampled"]
fs_kinematics = float(kinematic_data["FS_kinematics_sub"])
events = np.asarray(kinematic_data["EVENTS"],dtype=int).ravel()
event_times = np.asarray(kinematic_data["EVENT_TIMES"]).ravel()
spike_times = [np.sort(np.asarray(unit,dtype=float).ravel()) for unit in spike_data["spike_times"]]

print("Kinematics (samples, variables):", ...) # Student Code
print("number of Spike units:", ...) # Student Code
print("Sampling rates (Hz) of kinematics and subsampled kinematics", ...) # Student Code

In [ ]:
# TODO: Use Raster Plot to visualize spiking data of the first 100 second
# Hint: use matplotlib eventplot

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
... # Student Code
plt.show()

In [ ]:
# TODO: Visualize hand trajectories and speed of first 100 second
time_kinematics = np.arange(len(hand_kinematics))/fs_kinematics
speed = np.linalg.norm(hand_kinematics[:,2:4],axis=1)

fig,axs = plt.subplots(1,2,figsize=(11,4),dpi=120)
indices = ... # Student Code
axs[0].plot(...) # Student Code
axs[0].set(xlabel="X position (cm)",ylabel="Y position (cm)",title="Reach trajectories: first 100 second")
axs[0].set_aspect("equal",adjustable="box")
axs[1].plot(time_kinematics,speed)
axs[1].set(xlim=(0, 100), xlabel="Time (s)",ylabel="Hand speed (cm/s)",title="Hand speed")
fig.tight_layout()

## 2. Epoching and Trial Averaged Visualization

Use the provided `epoch_trials` helper to identify trials and their alignment events. Inspect the event codes and trial counts before averaging. In the aligned plots, time zero marks the selected event; negative times occur before it.

In [ ]:
from src.epoching import epoch_trials
"""
epoch_trials function return 4 arrays:
trial_events (n_trials, n_events): The event sequence of each trial
reach_direction (n_trials): The reach direction of each trial
align_time_lc (n_trials): Time when hand leaves center target for each trial
align_time_go (n_trials): Time when go cue occurs
"""
(   trial_events,
    reach_direction,
    align_time_lc,
    align_time_go
) = epoch_trials(events, event_times, hand_kinematics, fs_kinematics)

print("First 10 trial event sequences:\n", trial_events[:10])
print("Trials per target:", np.unique(reach_direction, return_counts=True))

**Question 2.1:** Look at the first 10 trial event sequences. Were the first 10 trials successful executions of the task? 

**Answer:** 

In [ ]:
# TODO: complete the trial-alignment helper; read the docstring for expectations

def trial_align_data(data, align_times, time_before, segment_length, fs):
    """
    Extract fixed-length segments of continuous data aligned to trial events.

    Parameters
    ----------
    data : 
        Time-series data with time along the first dimension.
    align_times : 
        Event times, in seconds, used to align each trial.
    time_before :
        Time, in seconds, to include before each alignment event.
    segment_length : 
        Total length, in seconds, of each desired extracted segment.
    fs : 
        Sampling rate of `data` in Hz.

    Returns
    -------
    trial_data : 
        Trial-aligned data with shape
        (n_trials, segment_length * fs, ...).
        Samples outside the recorded data range are filled with NaN.
    """
    data = np.asarray(data)
    n_samples = round(segment_length*fs)
    trial_data = np.full((len(align_times), n_samples) + data.shape[1:], np.nan)

    for trial,align_time in enumerate(align_times):
        ... # Student Code
    return trial_data

In [ ]:
# Speed aligned to movement onset and go-cue.
time_before = 1.5
segment_length = 3.0
trial_time_kin = np.arange(int(segment_length*fs_kinematics))/fs_kinematics-time_before
trial_speed_lc = trial_align_data(speed, align_time_lc, time_before, segment_length, fs_kinematics)
trial_speed_go = trial_align_data(speed, align_time_go, time_before, segment_length, fs_kinematics)

fig,axs = plt.subplots(1,2,figsize=(11,4),dpi=120,sharey=True)
for ax,trial_speed,title in zip(axs,(trial_speed_go,trial_speed_lc),("Go-cue", "Leaving center")):
    ax.plot(trial_time_kin, trial_speed[:75].T,alpha=0.2,linewidth=0.6)
    ax.plot(trial_time_kin, np.nanmean(trial_speed,axis=0),"k",linewidth=2.5,label="trial mean")
    ax.axvline(0,color="k",linestyle="--")
    ax.set(xlabel="Time from event (s)",ylabel="Hand speed (cm/s)",title=title)
    ax.legend()
fig.tight_layout()

**Question 2.2:** What are the similarities of these two trial-averaged speed profiles? What are the differences and why?

**Answer:** 

**Question 2.3:** What are the benefits and caveats of looking at trial-averaged speed profiles?

**Answer:**

In [ ]:
# TODO: complete the spike-alignment and binning helper; read the docstring for expectations

# Hint: for each trial, shift spike times relative to the alignment event,
# then count spikes falling inside each moving time window.

# Optional: there are many more ways to implement this function
# feel free to discard the template and write a more efficient one!

def trial_align_and_bin_spikes(spike_times, align_times, time_before, time_after, bin_width):
    """
    Bin spike times into trial-aligned firing rates.

    Parameters
    ----------
    spike_times :
        List of spike-time arrays, one array per neural unit.
    align_times :
        Event times, in seconds, used to align each trial.
    time_before :
        Time, in seconds, to include before each alignment event.
    time_after :
        Time, in seconds, to include after each alignment event.
    bin_width :
        Width of each firing-rate window, in seconds.

    Returns
    -------
    rates :
        Trial-aligned firing rates with shape
        (n_trials, n_bins, n_units).
    bin_times :
        Center time of each firing-rate window relative to the alignment event.
    """

    n_bins = round((time_before + time_after) / bin_width)
    bin_times = np.linspace(-time_before + bin_width/2, time_after - bin_width/2, n_bins)
    rates = np.zeros((len(align_times), n_bins, len(spike_times)))

    for trial, align_time in enumerate(align_times):
        for unit, times in enumerate(spike_times):

            relative_times = ... # Student Code

            for bin_idx, center in enumerate(bin_times):

                start = ... # Student Code
                stop = ... # Student Code

                in_window = ((...) & (...)) # Student Code

                rates[trial, bin_idx, unit] = ... # Student Code

    return rates, bin_times

Use the spike-binning helper to compare firing rates with hand speed around movement onset. Divide spike counts by the bin width in seconds to obtain Hz. Then compare direction-specific responses for units 2 and 34 using 25, 100, and 250 ms bins, keeping the alignment interval fixed.

In [ ]:
# Compare trial-averaged firing rates and peed, centered on "leaving center" state
trial_spike_rate,trial_time_spikes = trial_align_and_bin_spikes(spike_times,align_time_lc,1.5,1.5,0.05)
example_units = np.array([2,17,27,34])-1
fig,axs = plt.subplots(2,1,figsize=(10,6),dpi=120,sharex=True)
for unit in example_units:
    axs[0].plot(trial_time_spikes,trial_spike_rate[:,:,unit].mean(axis=0),label=f"Unit {unit+1}")
axs[0].set(ylabel="Firing rate (Hz)",title="Trial-averaged firing rates")
axs[0].legend()
axs[1].plot(trial_time_kin,trial_speed_lc.mean(axis=0),"k")
axs[1].set(xlabel="Time from leaving center (s)",ylabel="Speed (cm/s)",title="Trial-averaged hand speed")
for ax in axs:
    ax.axvline(0,color="k",linestyle="--")
fig.tight_layout()

In [ ]:
# Visualize direction-specific firing rates at three bin widths (25 ms, 100ms, 250ms)
# for two neurons unit 2 and unit 34.
directions = np.arange(1, 9)
colors = plt.get_cmap("tab10")(np.arange(8))
fig,axs = plt.subplots(2,3,figsize=(13,6),dpi=120,sharex=True)
for col,bin_width in enumerate((0.025,0.1,0.25)):
    rates,bin_times = trial_align_and_bin_spikes(spike_times,align_time_lc,1.5,1.5,bin_width)
    for row,unit in enumerate((1,33)):
        for direction in directions[::2]:
            axs[row,col].plot(bin_times,rates[reach_direction==direction,:,unit].mean(axis=0),
                              color=colors[direction-1],label=f"Target {direction}")
        axs[row,col].axvline(0,color="k",linestyle="--")
        axs[row,col].set(xlabel="Time from leaving center (s)",ylabel="Firing rate (Hz)",
                         title=f"Unit {unit+1}, {bin_width*1000:g} ms bins")
axs[0,0].legend(fontsize=8)
fig.tight_layout()

**Question 2.4:** Which example units increase or decrease their firing near movement onset? Does the timing of their activity match the hand-speed profile?

**Answer:** 

**Question 2.5:** How does bin width affect the smoothness and temporal detail of the direction-specific responses? Do units 2 and 34 respond most strongly to the same target directions?

**Answer:** 

## 3. LDA Prediction

Build an LDA decoder for reach direction using neural firing rates. For each trial, compute averaging firing rate over a time window around movement onset. Train the decoder on 50% of the trials and evaluate it on the remaining 50%. Then vary the window (for example, its start time, end time, or duration) and compare decoding performance while keeping the same train/test split. Finally, consider how performance changes when the feature window is restricted to only neural activity available before or up to movement onset (a causal window).

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from src.lda import plot_confusion, run_leave_one_out_classification

In [ ]:
# TODO: design the window to extract average firing rate, which is later fed into the lda predictor.
# Experiment with different window choice
def firing_rate_features(rates,bin_times,start,stop):
    """
    Extract featrue (one feature per unit) from trial-aligned firing rates within a specified time window.

    Parameters
    ----------
    rates :
        Firing rates with shape (n_trials, n_bins, n_units).
    bin_times :
        Time of each firing-rate bin relative to the alignment event.
    start :
        Start time of the averaging window, in seconds.
    stop :
        End time of the averaging window, in seconds.

    Returns
    -------
    features : 
        (n_trials, n_units).
    """
    window = ... # Student Code
    return ... # Student Code

classification_rates, classification_times = trial_align_and_bin_spikes(spike_times,align_time_lc,1.5,1.5,0.1) 

start = ... # Student Code
stop  = ... # Student Code
neural_features = firing_rate_features(classification_rates,
                                       classification_times,
                                       start,
                                       stop)

trial_indices = np.arange(len(reach_direction))
train_indices,test_indices = train_test_split(trial_indices,train_size=0.5,stratify=reach_direction,random_state=0)
decoder = LinearDiscriminantAnalysis()
decoder.fit(neural_features[train_indices],reach_direction[train_indices])
predicted_direction = decoder.predict(neural_features[test_indices])
plot_confusion(reach_direction[test_indices],predicted_direction,"Spikes: 50% held out")
plt.tight_layout()

**Question 3.1:** What windows did you try? If you are using a causal window length, how does that affect performance?

**Answer:** 

**Question 3.2:** Does a longer firing-rate window always improve decoding? If not, at what point does increasing the window stop improving estimates?

**Answer:** 

**Question 3.3:** Which target directions are most often confused by LDA? Are these errors mostly between nearby directions?

**Answer:** 

## 4. Evaluate what affects Decoding Accuracy

Use leave-one-out evaluation to study the effects of feature-window duration, ensemble size, and individual-unit selection. Each prediction must come from a model trained without that trial. For ensemble size, repeat the analysis with random subsets and inspect both the mean and spread of accuracy. The single-unit time-window analysis may take several minutes.

In [ ]:
# Exhaustive leave-one-out classification.
loo_prediction = run_leave_one_out_classification(neural_features,reach_direction)
plot_confusion(reach_direction,loo_prediction,"Spikes: leave-one-out")
plt.tight_layout()

In [ ]:
# TODO: Accuracy versus number of units, averaged over random subsets.
ensemble_features = firing_rate_features(classification_rates,classification_times,-0.5,0.5)
n_units = ensemble_features.shape[1]
unit_counts = np.arange(1, n_units+1, 3)
n_repeats = 10

subset_accuracy = np.empty((len(unit_counts),n_repeats))
for row,n_selected in tqdm(enumerate(unit_counts), total=len(unit_counts)):
    for repeat in range(n_repeats):
        # Hint: use rng to select a random subset of units of size n_selected
        units = rng.choice(...) # Student Code
        prediction = run_leave_one_out_classification(...) # Student Code
        subset_accuracy[row,repeat] = np.mean(prediction==reach_direction)
        
fig,ax = plt.subplots(figsize=(6,4),dpi=120)
mean_accuracy = subset_accuracy.mean(axis=1)
sd_accuracy = subset_accuracy.std(axis=1)
ax.plot(unit_counts,mean_accuracy)
ax.fill_between(unit_counts,mean_accuracy-sd_accuracy,mean_accuracy+sd_accuracy,alpha=0.2,label="±1 SD")
ax.axhline(1/8,color="k",linestyle="--",label="chance")
ax.set(xlabel="Number of units",ylabel="Leave-one-out accuracy",ylim=(0,1),title="Random neural ensembles")
ax.legend()
fig.tight_layout()

In [ ]:
# TODO: Single-unit decoding and its dependence on time.
# This is slow to run, may take 2~5 minutes

# Single unit decoding accuracy
single_unit_accuracy = np.empty(n_units)
for unit in range(n_units):
    prediction = run_leave_one_out_classification(ensemble_features[:,unit:unit+1],reach_direction)
    single_unit_accuracy[unit] = ... # Student Code

# Single unit decoding accuracy, dependence on time
window_starts = np.arange(-15,11)/10
unit_time_accuracy = np.empty((n_units,len(window_starts)))
for col, start in tqdm(enumerate(window_starts), total=len(window_starts)):
    # Hint: use firing_rate_features function we previous written on the defined segment
    features = ... # Student Code
    for unit in range(n_units):
        prediction = run_leave_one_out_classification(features[:,unit:unit+1],reach_direction)
        unit_time_accuracy[unit,col] = ... # Student Code


# Plotting
fig,axs = plt.subplots(1,2,figsize=(12,4),dpi=120)
axs[0].bar(np.arange(1, n_units+1),single_unit_accuracy)
axs[0].axhline(1/8,color="k",linestyle="--")
axs[0].set(xlabel="Unit ID",ylabel="Leave-one-out accuracy",title="Single units: −0.5 to 0.5 s")
image = axs[1].imshow(unit_time_accuracy,origin="lower",aspect="auto",cmap="hot",vmin=0,
                      extent=(window_starts[0]-0.05,window_starts[-1]+0.05,0.5,n_units+0.5))
axs[1].set(xlabel="Window start (s)",ylabel="Unit ID",title="Single units: 0.5-second windows")
fig.colorbar(image,ax=axs[1],label="Leave-one-out accuracy")
fig.tight_layout()


**Question 4.1:** When are individual units most informative about reach direction? Relate the time-dependent accuracy plot to the previous plot of firing vs. time (“Visualize direction-specific firing rates at three bin widths”)

**Answer:** 

**Question 4.2:** If you select the best window or ensemble size using these leave-one-out scores, why would you need a separate test set to report final performance?

**Answer:** 

## 5. Wiener Filter

The **Wiener filter** is a linear time-invariant filter. BMI applications typically use the discrete filter implementation. We will discuss the Wiener filter specifically in the context of BMI applications for predicting movement kinematics. The discrete Wiener filter models movement properties $Y(t)$ as a linear function of past neural activity $X(t)$:

$$
Y(t)=b+\sum_{i=-m}^{n} a(i)X(t-i)+\varepsilon(t)
$$

$$
Y=XA
$$

where $a(i)$ are filtering constants at time-lag $i$, $b$ is a vector of offsets, and $\varepsilon(t)$ is a noise term. The filter uses neural activity across time-lags $[-m,n]$. This can alternately be expressed in matrix form, where $X$ is a matrix of neural activity where each column is a different unit and rows are time-lagged data, and $A$ is a matrix of the corresponding filter coefficients.

Incorporating time-lagged data allows the Wiener filter to use the temporal dynamics of neural activity for prediction. Online BMI applications must be causal filters, using only data from the past to predict current/future events. However, open-loop off-line predictions can use non-causal filtering.

Wiener filters are best suited to continuous variable prediction in real-time BMI applications, since they are designed to predict time-series data. Wiener filters can be used to predict a variety of different movement variables, including kinematics or dynamics in joint or Cartesian space.

Training a Wiener filter requires setting the filtering constants (matrix $A$). When $X_{\text{train}}$ has full column rank, the coefficients that minimize least-squared error are given by

$$
A=\left(X_{\text{train}}^{T}X_{\text{train}}\right)^{-1}
X_{\text{train}}^{T}Y_{\text{train}}
$$

where $X_{\text{train}}$ is the neural design matrix (including time lags and an intercept column) and $Y_{\text{train}}$ is the matrix of corresponding kinematic training samples. In code, feel free to use `np.linalg.lstsq` rather than explicitly computing the inverse. This is equivalent to building a linear regression model between neural firing rates and kinematics.

In [ ]:
from src.filters import downsample_kin_spike, plot_velocity_prediction

In [ ]:
Y_kin, X_spikes, time_continuous = downsample_kin_spike(hand_kinematics, fs_kinematics, spike_times, bin_width=0.2, align_bins=True)

Fit a Wiener filter to predict x-velocity and y-velocity from the binned spike rates. The provided lag-matrix helper includes an intercept and orders neural samples from oldest to newest; align each row with the velocity at its newest sample. Compare 1 and 20 lags, then vary the number of lags from 1 to 40. These plots evaluate predictions on the same data used for fitting.

In [ ]:
# TODO: One-lag Wiener filter

def make_time_lagged_matrix(X,num_lags):
    if not 1<=num_lags<=len(X):
        raise ValueError("Number of lags must be between 1 and the number of samples.")
    n_rows = len(X)-num_lags+1
    return np.column_stack([X[lag:lag+n_rows] for lag in range(num_lags)]+[np.ones(n_rows)])

def train_wiener_filter(X,Y,num_lags):
    lagged_X = make_time_lagged_matrix(X,num_lags)
    return ... # Student Code

Y_velocity = Y_kin[:,2:4]
wiener_one = train_wiener_filter(...) # Student Code
velocity_wiener_one = make_time_lagged_matrix(X_spikes,1)@wiener_one 
r2_wiener_one = plot_velocity_prediction(time_continuous,Y_velocity,velocity_wiener_one,
                                         "One-lag Wiener filter: training-data predictions")

In [ ]:
# TODO: Twenty-lag Wiener filter.
wiener_lags = 20
wiener_weights = train_wiener_filter(...) # Student Code
velocity_wiener = make_time_lagged_matrix(X_spikes,wiener_lags)@wiener_weights 
r2_wiener = plot_velocity_prediction(time_continuous[wiener_lags-1:],Y_velocity[wiener_lags-1:],velocity_wiener,
                                     "20-lag Wiener filter: training-data predictions")

In [ ]:
# TODO: Wiener performance versus filter length.
lag_counts = np.arange(1,41)
wiener_r2_by_lag = np.empty((len(lag_counts),2))
for row,num_lags in enumerate(lag_counts):
    weights = train_wiener_filter(...) # Student Code
    prediction = make_time_lagged_matrix(X_spikes,num_lags)@weights 
    wiener_r2_by_lag[row] = r2_score(...) # Student Code

# plotting
fig,ax = plt.subplots(figsize=(6,4),dpi=120)
ax.plot(lag_counts,wiener_r2_by_lag[:,0],label="X velocity")
ax.plot(lag_counts,wiener_r2_by_lag[:,1],label="Y velocity")
ax.set(xlabel="Number of lags",ylabel="Training-data R²",title="Wiener filter performance")
ax.legend()
fig.tight_layout()

**Question 5.1:** How do the predicted velocity traces and R² values change when you add lags? With 200 ms bins, how far apart are the oldest and newest samples in a 20-lag feature vector?

**Answer:** 

**Question 5.2:** Why can training-data R² favor a more complex filter without demonstrating better predictions on new data? How would you split this continuous recording to evaluate model performance on test data while preventing features needed for assessing test data from being used in training data

**Answer:** 

## 6. Kalman Filter

The Kalman Filter predicts the state of a system by modeling the state dynamics and their relationship to observed variables. Like the Wiener filter, the Kalman Filter is a linear filter. However, the Kalman filter is a state-based filter where probabilistic models are used for prediction. The Wiener filter, in contrast, does not model an underlying state and its dynamics, but fits them empirically.

We will discuss the Kalman Filter specifically in the context of its use as a BMI decoder. When used as a BMI decoder, the Kalman Filter is used to predict 2D cursor kinematics from measured neural activity. The filter models the underlying dynamics of cursor movement (i.e. given a particular cursor position at time $t$, what is the most likely cursor position at $t+1$). It also models the relationship between measured neural activity and cursor movement. In order to train the Kalman filter decoder, data from actual arm movements is collected simultaneously with neural recordings. To test a Kalman filter decoder prior to its use as a BMI, held-out arm movement data is predicted from held-out neural recordings. The final filter estimate is a combination of estimates based on the dynamics model and neural activity measurements.

The KF assumes linear state evolution (1) and state observation models (2):

$$
x_t = A x_{t-1} + w_t
$$

$$
y_t = C x_t + q_t
$$

where $x_t$ represent the movement variables and $y_t$ represent recorded neural activity at time $t$, respectively (note that the notation convention is different from Wiener Filter). $w_t \sim N(0, W)$ and $q_t \sim N(0, Q)$ are noise terms. The filter is thus specified by the matrices $A$, $W$, $C$, and $Q$, which are estimated using training data.

Based on these models, the Kalman Filter recursively estimates the current movement variables $x_t$ based on both the previously estimated states $(x_{t-1}, \ldots, x_0)$ and currently observed neural activity $y_t$. 

Since we did not cover the detailed training process of Kalman filters, 
we do not ask you to complete any code in this section. 
However, try to go through and understand the code and answer the questions, 
which prepares you for the next section when we ask you about the role of each matrices in Kalman Filter.

In [ ]:
from src.filters import train_kalman_filter, run_kalman_forward

In [ ]:
# Kalman velocity predictions with a constant state for the firing-rate offset.
# use train_kalmand_filter and run_kalman_forward functions provided
X_kin, Y_spikes, time_continuous = downsample_kin_spike(hand_kinematics, fs_kinematics, spike_times, bin_width=0.2, align_bins=True)
X_velocity = X_kin[:,2:4]
kalman_states = np.column_stack((X_velocity,np.ones(len(X_velocity)))).T
A_k,W_k,C_k,Q_k = train_kalman_filter(Y_spikes.T,kalman_states)
velocity_kalman, _ = run_kalman_forward(A_k,W_k,C_k,Q_k,Y_spikes.T,kalman_states[:,0])
velocity_kalman = velocity_kalman[:2].T
r2_kalman = plot_velocity_prediction(time_continuous,X_velocity,velocity_kalman,
                                     "Kalman filter, 200 ms bins: training-data predictions")
print("Training-data R² (X, Y)")
print("Wiener, 1 lag:",r2_wiener_one)
print("Wiener, 20 lags:",r2_wiener)
print("Kalman:",r2_kalman)

**Question 6.1:** What roles do the state-transition matrix, observation matrix, process-noise covariance, and observation-noise covariance play in the Kalman filter? 

**Answer:** 
- A: 
- C: 
- W:
- Q:

**Question 6.2:** Compare the Kalman and Wiener velocity predictions, including their timing, smoothness, and R². What can you conclude from these training-data comparisons, and what still requires held-out evaluation?

**Answer:** 

## 7. Impulse Response Analysis

In the previous section, you will probably notice that even though Kalman filter (slightly) beats 1-lag Wiener filter, its R2 performance on velocity is not as good as 20-lag Wiener filter.However, maximizing agreement with the measured velocity is not the only objective in BMI decoder design. In this section, we will explore the benefits of Kalman filter as a state space model. 

We will now compare the dynamics of the Kalman and Wiener filters directly by examining their **impulse response**. This puts in a brief simulated pulse of input to a filter and studies the response output. For BMI, this is equivalent to putting in a single burst of neural firing and then observing the resulting kinematic trajectory.

Load your previously-saved Wiener filter and Kalman filter parameters. Compare the impulse response for the Wiener and Kalman filters. Generate a fake neural input for unit 1, active for 2 time-bins (400 ms total). Make a plot with:
- Neural input time-series
- x-velocity and y-velocity over time (overlaid traces on the same plot) for the Wiener filter
- x-position and y-position trajectory for the Wiener filter
- x-velocity and y-velocity  over time (overlaid traces on the same plot) for the Kalman filter
- x-position and y-position trajectory for the Kalman filter


In [ ]:
# TODO: Impulse responses relative to zero-input baselines remove fitted DC offsets.
simulation_bin = 0.2
time_sim = np.arange(int(30/simulation_bin))*simulation_bin
pulse_start = int(10/simulation_bin)
zero_input = np.zeros((len(time_sim),len(spike_times)))
initial_state = np.array([0.0,0.0,1.0])
wiener_baseline = make_time_lagged_matrix(zero_input,wiener_lags)@wiener_weights

def neural_pulse(unit):
    neural_input = zero_input.copy()
    neural_input[pulse_start:pulse_start+2,unit] = 10
    return neural_input

def kalman_impulse_response(neural_input,A=A_k,W=W_k):
    # hint: use run_kalman_forward function, similar to how its called above
    baseline, kalman_gains_baseline = run_kalman_forward(...) # Student Code
    response, kalman_gains_response = run_kalman_forward(...) # Student Code
    return (response[:2]-baseline[:2]).T

def filter_impulse_responses(unit):
    neural_input = neural_pulse(unit)
    wiener_velocity = make_time_lagged_matrix(neural_input,wiener_lags)@wiener_weights-wiener_baseline 
    kalman_velocity = kalman_impulse_response(neural_input) 
    return neural_input,wiener_velocity,kalman_velocity


neural_input,wiener_velocity,kalman_velocity = filter_impulse_responses(0)
wiener_position = np.cumsum(wiener_velocity,axis=0)*simulation_bin
kalman_position = np.cumsum(kalman_velocity,axis=0)*simulation_bin
fig = plt.figure(figsize=(12,8),dpi=120)
grid = fig.add_gridspec(3,3)
input_ax = fig.add_subplot(grid[0,:])
input_ax.plot(time_sim,neural_input[:,0])
input_ax.set(xlabel="Time (s)",ylabel="Firing rate (Hz)",title="Unit 1: 400 ms pulse")
for row,name,times,velocity,position in (
    (1,"Wiener",time_sim[wiener_lags-1:],wiener_velocity,wiener_position),
    (2,"Kalman",time_sim,kalman_velocity,kalman_position),
):
    ax = fig.add_subplot(grid[row,:2])
    ax.plot(times,velocity[:,0],label="X velocity")
    ax.plot(times,velocity[:,1],label="Y velocity")
    ax.axvline(10,color="k",linestyle="--")
    ax.set(xlabel="Time (s)",ylabel="Velocity (cm/s)",title=f"{name} impulse response")
    ax.legend()
    ax = fig.add_subplot(grid[row,2])
    ax.plot(position[:,0],position[:,1])
    ax.set(xlabel="X position (cm)",ylabel="Y position (cm)",title=f"{name} trajectory")
    ax.set_aspect("equal",adjustable="box")
fig.tight_layout()

Repeat the pulse experiment for the seven example units below. Compare trajectories relative to each filter's zero-input baseline, so fitted offsets do not dominate the response. Convert velocity to position by cumulatively summing and multiplying by the 0.2 s time step.

In [ ]:
# TODO: Impulse trajectories for seven example units.
simulation_units = np.array([1,10,18,20,32,35,38])-1
fig,axs = plt.subplots(1,2,figsize=(11,5),dpi=120)
for unit in simulation_units:
    _,wiener_velocity,kalman_velocity = filter_impulse_responses(unit)
    for ax,velocity in zip(axs,(wiener_velocity,kalman_velocity)):
        # hint: use np.cumsum
        position = ... # Student Code
        ax.plot(position[:,0],position[:,1],label=f"Unit {unit+1}")
for ax,name in zip(axs,("Wiener","Kalman")):
    ax.set(xlabel="X position (cm)",ylabel="Y position (cm)",title=f"{name}: impulse trajectories")
    ax.set_aspect("equal",adjustable="box")
    ax.legend(fontsize=8)
fig.tight_layout()

**Question 7.1:** What key difference do you notice between the Wiener filter and Kalman filter impulse responses?

**Answer:** 

**Question 7.2:** Why do each of the different units produce a notably different impulse response? Hint:
think about your analyses from one of the previous plots

**Answer:** 

## 8. Manipulation of Kalman Filter Matrices

Manipulate the $A$ and $W$ matrices in the Kalman filter to understand how they impact the impulse response. Define

$$
A_{\text{new}} =
\begin{bmatrix}
V_{xx} & V_{xy} & 0 \\
V_{yx} & V_{yy} & 0 \\
0 & 0 & 1
\end{bmatrix}
$$

and

$$
W_{\text{new}} = \operatorname{diag}(W) =
\begin{bmatrix}
W_{xx} & 0 & 0 \\
0 & W_{yy} & 0 \\
0 & 0 & 0
\end{bmatrix}.
$$

Note that $W_{\text{new}}$ is simply the diagonalized version of the learned $W$ matrix in the Kalman filter, meaning that all cross-terms are set to zero.

First, change the scaling of $A_{\text{new}}$ by making it a diagonal matrix, such that $V_{xy}=V_{yx}=0$, and set $V_{xx}=V_{yy}=\alpha$ for a range of $\alpha$ values from 0 to 1.2. Plot the impulse-response X-Y position trajectory for one example unit input for each scaling of $A$, with all trajectories shown on the same plot.

Next, change the cross-terms of $A_{\text{new}}$ while fixing $V_{xx}=V_{yy}=0.5$. First, set $V_{xy}=V_{yx}=\beta$ for a range of $\beta$ values from 0 to 0.5. Plot the impulse-response X-Y position trajectory for one example unit input for each value of $\beta$, with all trajectories on the same plot.

Then, set $V_{xy}=\beta$ and $V_{yx}=-\beta$ for a range of $\beta$ values from 0 to 0.5. Again, plot the impulse-response X-Y position trajectory for one example unit input for each value of $\beta$, with all trajectories shown on the same plot.

Finally, fix $A_{\text{new}}$ as a diagonal matrix with $V_{xy}=V_{yx}=0$ and $V_{xx}=V_{yy}=0.5$. Change the scaling of $W_{\text{new}}$ by setting

$$
W_{\text{new}} = \alpha W_{\text{new}}.
$$

For each scaling of $W$, plot the impulse-response Y position as a function of time for one example unit input, with all trajectories shown on the same plot.

In [ ]:
# Kalman state-transition scaling and cross-terms.
neural_input = neural_pulse(9)

W_diagonal = ... # Student Code
alphas = ... # Student Code
betas = ... # Student Code

fig,axs = plt.subplots(1,3,figsize=(14,4),dpi=120)
for alpha in alphas:
    A_new = ... # Student Code
    velocity = kalman_impulse_response(neural_input,A_new,W_diagonal)
    position = ... # Student Code
    axs[0].plot(position[:,0],position[:,1],label=f"α={alpha:g}")
for ax,sign in zip(axs[1:],(1,-1)):
    for beta in betas:
        A_new = ... # Student Code
        velocity = kalman_impulse_response(neural_input,A_new,W_diagonal) 
        position = ... # Student Code
        ax.plot(position[:,0],position[:,1],label=f"β={beta:g}")
for ax,title in zip(axs,("Diagonal A scaling","Same-sign cross-terms","Opposite-sign cross-terms")):
    ax.set(xlabel="X position (cm)",ylabel="Y position (cm)",title=title)
    ax.set_aspect("equal",adjustable="box")
    ax.legend(fontsize=8)
fig.tight_layout()

In [ ]:
# Kalman process-noise scaling.
A_new =... # Student Code
fig,ax = plt.subplots(figsize=(8,4),dpi=120)
for alpha in np.arange(6)/5:
    velocity = kalman_impulse_response(neural_input,A_new,alpha*W_diagonal)
    position = ... # Student Code
    ax.plot(time_sim,position[:,1],label=f"α={alpha:g}")
ax.set(xlabel="Time (s)",ylabel="Y position (cm)",title="Impulse response versus process-noise covariance")
ax.legend()
fig.tight_layout()

**Question 8.1:** How does increasing the diagonal scaling of A change the distance traveled and persistence of the impulse response? Why does A alone not determine the response when neural observations continue to update the state estimate?

**Answer:** 

**Question 8.2:** How do same-sign and opposite-sign cross-terms change trajectory direction or curvature? Explain how these terms couple X and Y velocity.

**Answer:** 

**Question 8.3:** With A fixed, how does scaling W change the Y-position response? Explain the effect in terms of the filter's relative trust in its dynamics model and neural observations.

**Answer:** 

## References

[1] A.L. Orsborn, H.G. Moorman, S.A. Overduin, M. M. Shanechi, D. Dimitrov, and J.M. Carmena (2014) Closed-loop decoder adaptation shapes neural plasticity for skillful neuroprosthetic control, Neuron 82, pp. 1380-1393

[2] B. B. Averbeck and D. Lee (2005) Effects of noise correlations on information encoding and decoding. J Neurophysiol 95, pp. 3633 – 3644. 
